# TopConNet v2 — AI-based LOS/NLOS classification
**Architecture:** Transformer Encoder on real GNSS time-series features  
**Key improvements over v1:**
- Real GNSS features fed to the model (no more random-noise images)
- IQR outlier removal + delta-feature engineering
- Latency-driven epoch windowing (single-epoch *and* multi-epoch)
- Lightweight Transformer Encoder (4 layers, 4 heads, d_model=64)
- Focal Loss + AdamW + cosine LR warmup
- Temperature-scaled calibrated probabilities + ROC-tuned threshold
- Temporal smoothing for 1 Hz output fusion


In [ ]:
# Install dependencies (run once)
!pip install georinex xarray netCDF4 scikit-learn -q


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import georinex as gr
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, f1_score)
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
class Config:
    # Data
    FILE_PATH      = '/content/GNSS/rover_ublox.obs'  # <-- update path
    GNSS_FREQ_HZ   = 10          # measurement rate
    OUTPUT_HZ      = 1           # desired positioning output rate
    WINDOW_SIZE    = GNSS_FREQ_HZ // OUTPUT_HZ   # epochs per window (10)
    STRIDE         = 1           # sliding window stride
    LOS_NLOS_THRESHOLD = 2.0     # SNR residual threshold (dB-Hz)
    TEST_SIZE      = 0.15
    VAL_SIZE       = 0.15

    # Model (Transformer)
    N_FEATURES     = 7           # SNR, elev, Δsnr, Δelev, Δphase, sin_az, cos_az
    D_MODEL        = 64
    N_HEADS        = 4
    N_LAYERS       = 4
    D_FF           = 256
    DROPOUT        = 0.1
    NUM_CLASSES    = 2           # LOS=0, NLOS=1  (extend to 3 for multipath)

    # Training
    EPOCHS         = 30
    BATCH_SIZE     = 64
    LR             = 1e-3
    WEIGHT_DECAY   = 1e-4
    FOCAL_GAMMA    = 2.0
    LABEL_SMOOTH   = 0.1

    # Inference
    DEFAULT_THRESHOLD = 0.5      # updated by ROC tuning after training

cfg = Config()
print("Config loaded.")
print(f"  Window size  : {cfg.WINDOW_SIZE} epochs  ({cfg.WINDOW_SIZE/cfg.GNSS_FREQ_HZ:.1f} s)")


In [ ]:
# ── Step 1 : Data loading and feature engineering ────────────────────────────

def load_rinex_features(file_path: str) -> pd.DataFrame:
    """
    Load RINEX observation file and extract per-SV feature matrix.
    Returns a DataFrame with columns:
        time, sv, snr, elevation, azimuth, carrier_phase (if available)
    """
    obs = gr.load(file_path)
    df  = obs.to_dataframe().reset_index()

    # ── Identify SNR column ─────────────────────────────────────────────────
    snr_cols = [c for c in df.columns
                if isinstance(c, str) and c.startswith('S') and len(c) <= 3]
    if not snr_cols:
        raise ValueError("No SNR column found. Check RINEX file.")
    snr_col = snr_cols[0]
    print(f"SNR column  : {snr_col}")

    # ── Identify carrier-phase column (L1C, L1, …) ──────────────────────────
    phase_cols = [c for c in df.columns
                  if isinstance(c, str) and c.startswith('L') and len(c) <= 4]
    phase_col = phase_cols[0] if phase_cols else None
    print(f"Phase column: {phase_col}")

    # ── Select & rename ──────────────────────────────────────────────────────
    cols = {'time': 'time', 'sv': 'sv', snr_col: 'snr'}
    if 'elevation' in df.columns:
        cols['elevation'] = 'elevation'
    if 'azimuth' in df.columns:
        cols['azimuth'] = 'azimuth'
    if phase_col:
        cols[phase_col] = 'carrier_phase'

    df = df[list(cols.keys())].rename(columns=cols)
    df = df.dropna(subset=['snr']).copy()
    df['time'] = pd.to_datetime(df['time'])
    df = df.sort_values(['sv', 'time']).reset_index(drop=True)
    print(f"Raw samples : {len(df):,}  across {df['sv'].nunique()} SVs")
    return df


def clean_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    IQR-based outlier removal per SV + fill missing elevation/azimuth.
    """
    clean_frames = []
    for sv, grp in df.groupby('sv'):
        grp = grp.copy()

        # SNR IQR clip
        q1, q3 = grp['snr'].quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 3.0 * iqr, q3 + 3.0 * iqr
        grp = grp[(grp['snr'] >= lo) & (grp['snr'] <= hi)]

        # Elevation / azimuth fill
        if 'elevation' not in grp.columns:
            grp['elevation'] = 45.0          # fallback mid-sky
        else:
            grp['elevation'] = grp['elevation'].fillna(grp['elevation'].median())

        if 'azimuth' not in grp.columns:
            grp['azimuth'] = 0.0
        else:
            grp['azimuth'] = grp['azimuth'].fillna(0.0)

        if 'carrier_phase' not in grp.columns:
            grp['carrier_phase'] = np.nan

        clean_frames.append(grp)

    df_clean = pd.concat(clean_frames).sort_values(['sv', 'time']).reset_index(drop=True)
    print(f"After IQR cleaning: {len(df_clean):,} samples")
    return df_clean


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add delta features (rate-of-change) and azimuth encoding.
    Features: snr, elevation, Δsnr, Δelevation, Δcarrier_phase, sin_az, cos_az
    """
    frames = []
    for sv, grp in df.groupby('sv'):
        grp = grp.copy()

        # Delta features (difference between consecutive epochs)
        grp['delta_snr']   = grp['snr'].diff().fillna(0)
        grp['delta_elev']  = grp['elevation'].diff().fillna(0)

        if grp['carrier_phase'].notna().sum() > 5:
            grp['delta_phase'] = grp['carrier_phase'].diff().fillna(0)
        else:
            grp['delta_phase'] = 0.0

        # Circular azimuth encoding (avoids 0/360 discontinuity)
        az_rad = np.deg2rad(grp['azimuth'])
        grp['sin_az'] = np.sin(az_rad)
        grp['cos_az'] = np.cos(az_rad)

        # Per-SV normalization of SNR and elevation
        grp['snr_norm']  = (grp['snr']       - grp['snr'].mean())       / (grp['snr'].std()       + 1e-8)
        grp['elev_norm'] = (grp['elevation']  - grp['elevation'].mean()) / (grp['elevation'].std() + 1e-8)

        frames.append(grp)

    df_feat = pd.concat(frames).sort_values(['sv', 'time']).reset_index(drop=True)
    print(f"Feature matrix shape: {df_feat.shape}")
    return df_feat


# ── LABELLING ────────────────────────────────────────────────────────────────
def generate_labels(df: pd.DataFrame, threshold: float = 2.0) -> pd.DataFrame:
    """
    Label LOS/NLOS using SNR residual vs. a per-SV rolling median baseline.
    residual = |SNR - rolling_median(SNR, 20)|
    label = 1 (NLOS) if residual > threshold, else 0 (LOS)
    """
    frames = []
    for sv, grp in df.groupby('sv'):
        grp = grp.copy()
        grp['snr_baseline'] = grp['snr'].rolling(20, min_periods=3).median().bfill()
        grp['snr_residual'] = (grp['snr'] - grp['snr_baseline']).abs()
        grp['label'] = (grp['snr_residual'] > threshold).astype(int)
        frames.append(grp)

    df_lab = pd.concat(frames).reset_index(drop=True)
    los_pct  = (df_lab['label'] == 0).mean() * 100
    nlos_pct = (df_lab['label'] == 1).mean() * 100
    print(f"Label distribution → LOS: {los_pct:.1f}%  NLOS: {nlos_pct:.1f}%")
    return df_lab


FEATURE_COLS = ['snr_norm', 'elev_norm', 'delta_snr', 'delta_elev',
                'delta_phase', 'sin_az', 'cos_az']

print("Data utilities defined.")


In [ ]:
# ── Step 2 : Epoch-windowing Dataset ─────────────────────────────────────────

class GNSSWindowDataset(Dataset):
    """
    Sliding-window dataset over GNSS time-series.

    Each sample is a tensor of shape (window_size, n_features)
    representing `window_size` consecutive epochs for one SV.
    The label is the MAJORITY vote of all epoch labels in the window.

    For SINGLE-EPOCH mode, set window_size=1 and stride=1.
    """

    def __init__(self, df: pd.DataFrame,
                 feature_cols: list,
                 window_size: int = 10,
                 stride: int = 1):
        self.windows = []
        self.labels  = []

        for sv, grp in df.groupby('sv'):
            grp = grp.reset_index(drop=True)
            feats  = grp[feature_cols].values.astype(np.float32)
            labels = grp['label'].values

            for start in range(0, len(grp) - window_size + 1, stride):
                end   = start + window_size
                win   = feats[start:end]          # (window_size, n_features)
                # Majority vote: label the window NLOS if >50% epochs are NLOS
                lab   = int(labels[start:end].mean() >= 0.5)
                self.windows.append(win)
                self.labels.append(lab)

        self.windows = np.array(self.windows, dtype=np.float32)
        self.labels  = np.array(self.labels,  dtype=np.int64)
        print(f"Dataset: {len(self.windows):,} windows | "
              f"LOS={( self.labels==0).sum():,}  NLOS={(self.labels==1).sum():,}")

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.windows[idx])   # (W, F)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


def make_weighted_sampler(labels: np.ndarray) -> WeightedRandomSampler:
    """
    Class-balanced sampler to handle LOS/NLOS imbalance.
    """
    class_counts = np.bincount(labels)
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[labels]
    return WeightedRandomSampler(
        weights=torch.from_numpy(sample_weights).float(),
        num_samples=len(sample_weights),
        replacement=True
    )

print("Dataset class defined.")


In [ ]:
# ── Step 3 : TopConNet — Transformer Encoder ─────────────────────────────────

class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding for epoch positions."""
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() *
                        -(np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))   # (1, max_len, d_model)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class TopConNet(nn.Module):
    """
    Lightweight Transformer Encoder for GNSS LOS/NLOS classification.

    Input  : (batch, window_size, n_features)
    Output : (batch, num_classes)  — raw logits

    Architecture:
      feature projection → positional encoding →
      N × TransformerEncoderLayer → CLS token pool → classifier head
    """

    def __init__(self,
                 n_features:  int = 7,
                 d_model:     int = 64,
                 n_heads:     int = 4,
                 n_layers:    int = 4,
                 d_ff:        int = 256,
                 dropout:     float = 0.1,
                 num_classes: int = 2):
        super().__init__()

        # 1. Project raw features → d_model
        self.input_proj = nn.Sequential(
            nn.Linear(n_features, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
        )

        # 2. Learnable CLS token (aggregates sequence info for classification)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        # 3. Positional encoding
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)

        # 4. Transformer encoder stack
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,          # Pre-LN for training stability
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # 5. Classification head (CLS token → class logits)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, num_classes),
        )

    def forward(self, x):
        # x: (B, W, F)
        B, W, _ = x.shape

        # Project features
        x = self.input_proj(x)                     # (B, W, d_model)

        # Prepend CLS token
        cls = self.cls_token.expand(B, -1, -1)     # (B, 1, d_model)
        x   = torch.cat([cls, x], dim=1)           # (B, W+1, d_model)

        # Add positional encoding
        x = self.pos_enc(x)

        # Transformer
        x = self.transformer(x)                    # (B, W+1, d_model)

        # Use CLS token output for classification
        cls_out = x[:, 0, :]                       # (B, d_model)
        logits  = self.classifier(cls_out)         # (B, num_classes)
        return logits

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Quick architecture check
_model = TopConNet(
    n_features  = cfg.N_FEATURES,
    d_model     = cfg.D_MODEL,
    n_heads     = cfg.N_HEADS,
    n_layers    = cfg.N_LAYERS,
    d_ff        = cfg.D_FF,
    dropout     = cfg.DROPOUT,
    num_classes = cfg.NUM_CLASSES,
)
print(f"TopConNet parameters: {_model.count_params():,}")

# Test forward pass
_x = torch.randn(4, cfg.WINDOW_SIZE, cfg.N_FEATURES)
_y = _model(_x)
print(f"Input shape : {_x.shape}  →  Output shape: {_y.shape}")
del _model, _x, _y


In [ ]:
# ── Focal Loss + Label Smoothing ─────────────────────────────────────────────

class FocalLoss(nn.Module):
    """
    Focal Loss for binary/multi-class classification.
    Reduces weight of easy examples; focuses on hard NLOS edge cases.
    FL(p) = -(1 - p_t)^gamma * log(p_t)
    """
    def __init__(self, gamma: float = 2.0,
                 label_smoothing: float = 0.1,
                 weight: torch.Tensor = None):
        super().__init__()
        self.gamma           = gamma
        self.label_smoothing = label_smoothing
        self.weight          = weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        # Label smoothing
        num_classes = logits.size(1)
        with torch.no_grad():
            smooth_targets = torch.zeros_like(logits)
            smooth_targets.fill_(self.label_smoothing / (num_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1),
                                    1.0 - self.label_smoothing)

        log_prob = torch.log_softmax(logits, dim=-1)
        prob     = log_prob.exp()

        # p_t for the true class
        p_t = (prob * smooth_targets).sum(dim=-1)

        focal_weight = (1 - p_t) ** self.gamma
        loss = -(focal_weight * (log_prob * smooth_targets).sum(dim=-1))

        if self.weight is not None:
            class_w = self.weight[targets]
            loss = loss * class_w

        return loss.mean()

print("FocalLoss defined.")


In [ ]:
# ── Step 4 : Training and validation loop ────────────────────────────────────

def train_epoch(model, loader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * y.size(0)
        preds   = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total   += y.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss   = criterion(logits, y)
        total_loss += loss.item() * y.size(0)
        probs  = torch.softmax(logits, dim=1)[:, 1]   # P(NLOS)
        preds  = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total   += y.size(0)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(y.cpu().numpy())

    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    auc = roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else 0.0
    return total_loss / total, correct / total, auc, all_probs, all_labels


def train(model, train_loader, val_loader, cfg, device, class_weights=None):
    """Full training loop with cosine LR warmup and early stopping."""
    weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)                     if class_weights is not None else None

    criterion = FocalLoss(gamma=cfg.FOCAL_GAMMA,
                          label_smoothing=cfg.LABEL_SMOOTH,
                          weight=weight_tensor)

    optimizer = optim.AdamW(model.parameters(),
                            lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)

    total_steps = len(train_loader) * cfg.EPOCHS
    scheduler   = CosineAnnealingWarmRestarts(optimizer, T_0=total_steps // 3)

    history = {'train_loss': [], 'val_loss': [],
               'train_acc':  [], 'val_acc': [], 'val_auc': []}
    best_auc, best_state, patience, patience_limit = 0.0, None, 0, 8

    for epoch in range(1, cfg.EPOCHS + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion,
                                       optimizer, scheduler, device)
        va_loss, va_acc, va_auc, _, _ = eval_epoch(model, val_loader,
                                                     criterion, device)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(va_acc)
        history['val_auc'].append(va_auc)

        print(f"Epoch {epoch:>3}/{cfg.EPOCHS}  "
              f"train_loss={tr_loss:.4f}  train_acc={tr_acc*100:.2f}%  "
              f"val_loss={va_loss:.4f}  val_acc={va_acc*100:.2f}%  "
              f"val_auc={va_auc:.4f}")

        if va_auc > best_auc:
            best_auc   = va_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience   = 0
        else:
            patience += 1
            if patience >= patience_limit:
                print(f"  Early stopping at epoch {epoch}  (best AUC={best_auc:.4f})")
                break

    model.load_state_dict(best_state)
    print(f"\nTraining complete. Best val AUC: {best_auc:.4f}")
    return history

print("Training utilities defined.")


In [ ]:
# ── Step 5 : Calibration and threshold tuning ────────────────────────────────

class TemperatureScaling(nn.Module):
    """
    Post-hoc probability calibration.
    Learns a single temperature T on the validation set.
    Calibrated logits = logits / T
    """
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, x):
        logits = self.model(x)
        return logits / self.temperature

    def fit(self, val_loader, device, n_iter=50, lr=0.01):
        self.to(device)
        optimizer = optim.LBFGS([self.temperature], lr=lr, max_iter=50)
        nll_criterion = nn.CrossEntropyLoss()

        # Collect all logits from the base model
        all_logits, all_labels = [], []
        self.model.eval()
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device)
                all_logits.append(self.model(x).cpu())
                all_labels.append(y)
        logits_t = torch.cat(all_logits).to(device)
        labels_t = torch.cat(all_labels).to(device)

        def eval_fn():
            optimizer.zero_grad()
            loss = nll_criterion(logits_t / self.temperature, labels_t)
            loss.backward()
            return loss
        optimizer.step(eval_fn)
        print(f"Calibrated temperature: {self.temperature.item():.4f}")


def tune_threshold_roc(probs: np.ndarray, labels: np.ndarray) -> float:
    """
    Find the probability threshold that maximises F1 on validation set.
    """
    fpr, tpr, thresholds = roc_curve(labels, probs)
    f1_scores = []
    for t in thresholds:
        preds = (probs >= t).astype(int)
        f1_scores.append(f1_score(labels, preds, zero_division=0))
    best_idx = int(np.argmax(f1_scores))
    best_t   = float(thresholds[best_idx])
    print(f"Optimal threshold (max F1={f1_scores[best_idx]:.4f}): {best_t:.4f}")
    return best_t

print("Calibration utilities defined.")


In [ ]:
# ── Step 6 : Temporal smoothing for 1 Hz output fusion ───────────────────────

def temporal_smooth_probs(probs: np.ndarray,
                           window: int = 3,
                           mode: str = 'majority') -> np.ndarray:
    """
    Smooth per-epoch NLOS probabilities into a stable output stream.

    Parameters
    ----------
    probs  : (N,) array of P(NLOS) per epoch
    window : smoothing window (epochs)
    mode   : 'ema'       – exponential moving average
             'majority'  – rolling majority vote on binary decisions
             'mean'      – rolling mean of probabilities

    Returns
    -------
    smoothed : (N,) array
    """
    if mode == 'ema':
        alpha = 2 / (window + 1)
        smoothed = np.zeros_like(probs)
        smoothed[0] = probs[0]
        for i in range(1, len(probs)):
            smoothed[i] = alpha * probs[i] + (1 - alpha) * smoothed[i - 1]
        return smoothed

    elif mode == 'mean':
        series = pd.Series(probs)
        return series.rolling(window, min_periods=1, center=True).mean().values

    elif mode == 'majority':
        series = pd.Series((probs >= 0.5).astype(int))
        return series.rolling(window, min_periods=1, center=True).mean().values

    else:
        raise ValueError(f"Unknown mode: {mode}")

print("Temporal smoothing defined.")


In [ ]:
# ── Evaluation and visualisation helpers ─────────────────────────────────────

def evaluate_model(model, test_loader, threshold: float, device,
                   class_names=('LOS', 'NLOS')):
    """Full evaluation: accuracy, AUC, F1, confusion matrix."""
    model.eval()
    all_probs, all_preds, all_labels = [], [], []

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            probs  = torch.softmax(logits, dim=1)[:, 1]
            all_probs.append(probs.cpu().numpy())
            all_labels.append(y.cpu().numpy())

    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    all_preds  = (all_probs >= threshold).astype(int)

    acc = (all_preds == all_labels).mean() * 100
    auc = roc_auc_score(all_labels, all_probs)
    f1  = f1_score(all_labels, all_preds)

    print(f"\n{'='*55}")
    print(f"  Test accuracy : {acc:.2f}%")
    print(f"  ROC-AUC       : {auc:.4f}")
    print(f"  F1 (NLOS)     : {f1:.4f}")
    print(f"  Threshold     : {threshold:.4f}")
    print(f"{'='*55}")
    print(classification_report(all_labels, all_preds,
                                 target_names=class_names))
    return all_probs, all_labels


def plot_training(history: dict):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(history['train_loss'], label='train')
    axes[0].plot(history['val_loss'],   label='val')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)

    axes[1].plot([a*100 for a in history['train_acc']], label='train')
    axes[1].plot([a*100 for a in history['val_acc']],   label='val')
    axes[1].set_title('Accuracy (%)'); axes[1].legend(); axes[1].grid(True)
    axes[1].yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

    axes[2].plot(history['val_auc'])
    axes[2].set_title('Val ROC-AUC'); axes[2].grid(True)

    plt.tight_layout(); plt.show()


def plot_confusion(all_labels, all_preds, class_names=('LOS', 'NLOS')):
    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names)
    ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title('Confusion Matrix')
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, str(cm[i, j]),
                    ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
    plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()


def plot_roc(all_labels, all_probs):
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    auc = roc_auc_score(all_labels, all_probs)
    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, label=f'AUC = {auc:.4f}')
    plt.plot([0,1],[0,1],'--', color='gray')
    plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC Curve')
    plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()


def plot_predictions_timeline(probs, labels, smoothed_probs, threshold):
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

    x = np.arange(len(probs))

    axes[0].plot(x, probs, color='steelblue', alpha=0.4, linewidth=0.8, label='raw P(NLOS)')
    axes[0].plot(x, smoothed_probs, color='tomato', linewidth=1.5, label='smoothed')
    axes[0].axhline(threshold, color='orange', linestyle='--', linewidth=1, label='threshold')
    axes[0].set_ylabel('P(NLOS)'); axes[0].legend(loc='upper right'); axes[0].grid(True)
    axes[0].set_title('NLOS probability over time')

    colors = ['steelblue' if l == 0 else 'tomato' for l in labels]
    axes[1].scatter(x, labels, c=colors, s=4, label='true label')
    preds = (smoothed_probs >= threshold).astype(int)
    wrong = np.where(preds != labels)[0]
    axes[1].scatter(wrong, labels[wrong], c='gold', s=16, marker='x', label='error')
    axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(['LOS', 'NLOS'])
    axes[1].set_xlabel('Epoch'); axes[1].legend(loc='upper right'); axes[1].grid(True)
    axes[1].set_title('Ground truth vs smoothed prediction')

    plt.tight_layout(); plt.show()

print("Evaluation helpers defined.")


In [ ]:
# ── MAIN : run the full pipeline ─────────────────────────────────────────────

def main():
    print("="*60)
    print("TopConNet v2 — LOS/NLOS Classification Pipeline")
    print("="*60)

    # ── 1. Load & clean ─────────────────────────────────────────────────────
    print("\n[1/6] Loading RINEX file...")
    df_raw  = load_rinex_features(cfg.FILE_PATH)
    df_cln  = clean_features(df_raw)
    df_feat = engineer_features(df_cln)
    df_lab  = generate_labels(df_feat, threshold=cfg.LOS_NLOS_THRESHOLD)

    # ── 2. Build datasets ────────────────────────────────────────────────────
    print("\n[2/6] Building windowed datasets...")
    # multi-epoch windows (primary model)
    full_ds = GNSSWindowDataset(df_lab, FEATURE_COLS,
                                 window_size=cfg.WINDOW_SIZE, stride=cfg.STRIDE)

    # indices split (stratified)
    idx = np.arange(len(full_ds))
    lbl = full_ds.labels
    idx_tr, idx_tmp, lbl_tr, lbl_tmp = train_test_split(
        idx, lbl, test_size=cfg.TEST_SIZE + cfg.VAL_SIZE,
        stratify=lbl, random_state=SEED)
    idx_va, idx_te = train_test_split(
        idx_tmp, test_size=cfg.TEST_SIZE / (cfg.TEST_SIZE + cfg.VAL_SIZE),
        stratify=lbl_tmp, random_state=SEED)

    from torch.utils.data import Subset
    train_ds = Subset(full_ds, idx_tr)
    val_ds   = Subset(full_ds, idx_va)
    test_ds  = Subset(full_ds, idx_te)
    print(f"  train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}")

    # class-balanced sampler for training
    sampler = make_weighted_sampler(lbl[idx_tr])

    train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, sampler=sampler)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=cfg.BATCH_SIZE, shuffle=False)

    # ── 3. Build model ───────────────────────────────────────────────────────
    print("\n[3/6] Initialising TopConNet...")
    model = TopConNet(
        n_features  = cfg.N_FEATURES,
        d_model     = cfg.D_MODEL,
        n_heads     = cfg.N_HEADS,
        n_layers    = cfg.N_LAYERS,
        d_ff        = cfg.D_FF,
        dropout     = cfg.DROPOUT,
        num_classes = cfg.NUM_CLASSES,
    ).to(DEVICE)
    print(f"  Parameters  : {model.count_params():,}")

    # ── 4. Train ─────────────────────────────────────────────────────────────
    print("\n[4/6] Training...")
    # class weights for focal loss
    class_counts = np.bincount(lbl[idx_tr])
    class_weights = len(lbl[idx_tr]) / (cfg.NUM_CLASSES * class_counts)
    history = train(model, train_loader, val_loader, cfg, DEVICE, class_weights)

    # ── 5. Calibrate probabilities ────────────────────────────────────────────
    print("\n[5/6] Temperature calibration + threshold tuning...")
    cal_model = TemperatureScaling(model)
    cal_model.fit(val_loader, DEVICE)

    # collect val probs for threshold tuning
    _, _, _, val_probs, val_labels = eval_epoch(
        cal_model, val_loader,
        FocalLoss(cfg.FOCAL_GAMMA, cfg.LABEL_SMOOTH), DEVICE)
    best_threshold = tune_threshold_roc(val_probs, val_labels)
    cfg.DEFAULT_THRESHOLD = best_threshold

    # ── 6. Evaluate on test set ───────────────────────────────────────────────
    print("\n[6/6] Final evaluation on test set...")
    test_probs, test_labels = evaluate_model(
        cal_model, test_loader, best_threshold, DEVICE)

    # temporal smoothing
    smoothed = temporal_smooth_probs(test_probs, window=3, mode='ema')

    # ── Plots ─────────────────────────────────────────────────────────────────
    plot_training(history)
    test_preds = (test_probs >= best_threshold).astype(int)
    plot_confusion(test_labels, test_preds)
    plot_roc(test_labels, test_probs)
    plot_predictions_timeline(test_probs, test_labels, smoothed, best_threshold)

    # ── Save model ────────────────────────────────────────────────────────────
    save_path = '/content/topconnet_v2.pt'
    torch.save({'model_state': cal_model.state_dict(),
                'config': vars(cfg),
                'threshold': best_threshold}, save_path)
    print(f"\nModel saved → {save_path}")

    return cal_model, best_threshold


if __name__ == '__main__':
    model, threshold = main()


In [ ]:
# ── Bonus: Single-epoch baseline comparison ───────────────────────────────────
# Your senior requested comparing single-epoch vs multi-epoch performance.

def run_single_epoch_baseline(df_lab):
    """
    Train and evaluate the same architecture on single-epoch inputs (window=1).
    Useful to quantify latency vs accuracy trade-off.
    """
    print("\n--- Single-epoch baseline ---")
    se_ds = GNSSWindowDataset(df_lab, FEATURE_COLS, window_size=1, stride=1)

    idx   = np.arange(len(se_ds))
    lbl   = se_ds.labels
    i_tr, i_tmp, l_tr, _ = train_test_split(idx, lbl, test_size=0.3,
                                             stratify=lbl, random_state=SEED)
    i_va, i_te           = train_test_split(i_tmp, test_size=0.5,
                                             stratify=_,  random_state=SEED)

    from torch.utils.data import Subset
    tr_l = DataLoader(Subset(se_ds, i_tr), batch_size=cfg.BATCH_SIZE,
                      sampler=make_weighted_sampler(l_tr))
    va_l = DataLoader(Subset(se_ds, i_va), batch_size=cfg.BATCH_SIZE)
    te_l = DataLoader(Subset(se_ds, i_te), batch_size=cfg.BATCH_SIZE)

    se_model = TopConNet(n_features=cfg.N_FEATURES, d_model=cfg.D_MODEL,
                         n_heads=cfg.N_HEADS, n_layers=cfg.N_LAYERS,
                         d_ff=cfg.D_FF, dropout=cfg.DROPOUT,
                         num_classes=cfg.NUM_CLASSES).to(DEVICE)

    cfg_se = Config()
    cfg_se.EPOCHS = 20            # fewer epochs for quick comparison
    class_w = len(l_tr) / (cfg.NUM_CLASSES * np.bincount(l_tr))
    train(se_model, tr_l, va_l, cfg_se, DEVICE, class_w)

    _, _, _, vp, vl = eval_epoch(se_model, va_l,
                                  FocalLoss(cfg.FOCAL_GAMMA, cfg.LABEL_SMOOTH),
                                  DEVICE)
    thr = tune_threshold_roc(vp, vl)
    evaluate_model(se_model, te_l, thr, DEVICE)

# Uncomment to run (requires df_lab from main()):
# run_single_epoch_baseline(df_lab)
print("Single-epoch baseline function ready — uncomment the last line to run.")


In [ ]:
# ── Inference helper (production use) ────────────────────────────────────────

def predict_window(model, window_np: np.ndarray,
                   threshold: float = 0.5,
                   device: torch.device = DEVICE):
    """
    Classify a single window of GNSS observations.

    Parameters
    ----------
    model      : trained TopConNet (with temperature scaling)
    window_np  : numpy array of shape (window_size, n_features)
    threshold  : P(NLOS) threshold
    device     : torch device

    Returns
    -------
    label : 0=LOS, 1=NLOS
    prob  : P(NLOS) ∈ [0, 1]
    """
    model.eval()
    x = torch.from_numpy(window_np.astype(np.float32)).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        prob   = torch.softmax(logits, dim=1)[0, 1].item()
    label = int(prob >= threshold)
    return label, prob


# Example (random window):
# label, prob = predict_window(model, np.random.randn(cfg.WINDOW_SIZE, cfg.N_FEATURES))
# print(f'Predicted: {"NLOS" if label else "LOS"}  P(NLOS)={prob:.4f}')
print("Inference helper defined.")
